In [19]:
import arff
import pandas as pd

# загружаем данные
df = pd.read_parquet("c:/Personal/Science/Data/Binary_classification/GermanCredit.pq")
pd.set_option("display.max_columns", None)
df.head()

,Unnamed: 0,acc_status,acc_duration,credit_history,purpose,credit_amount,savings_acc,employment_time,installment_rate,gender,other_debtors,residence_time,property,age,installment_plans,housing,num_credits_at_bank,job,num_people_maintenance,phone,foreigner,target
0,600,dm_0_to_200,7,credit_paid_so_far,radio_or_tv,2329,dm_less_100,less_one,1,female,guarantor,1,real_estate,45,none,own,1,skilled_employee_or_official,1,no,yes,1
1,80,no_acc,24,credit_paid_so_far,radio_or_tv,5943,unknown_or_no_savings,less_one,1,female,none,1,car_or_other,44,none,own,2,skilled_employee_or_official,1,yes,yes,2
2,158,dm_0_to_200,24,credit_paid_so_far,car_new,3512,dm_100_to_500,four_to_seven,2,male,none,3,car_or_other,38,bank,own,2,skilled_employee_or_official,1,yes,yes,1
3,423,no_acc,21,delay_in_past_credit_payment,furniture_or_equipment,1591,dm_100_to_500,four_to_seven,4,male,none,3,real_estate,34,none,own,2,self_or_highly_qualified_employee,1,no,yes,1
4,747,dm_less_than_0,12,credit_paid_so_far,car_new,1274,dm_less_100,less_one,3,female,none,1,real_estate,37,none,own,1,unskilled_resident,1,no,yes,2


In [21]:
from typing import List, Any
import numpy as np

def clean_dataset(
    df: pd.DataFrame,
    cols_to_drop: List[Any] = None,
    missing_values: List[Any] = None
) -> pd.DataFrame:
    """
    Предобработка датафрейма с учетом пользовательских меток пропусков:
    1. Заполняет пропущенные значения
       - Модой для категориальных/текстовых признаков.
       - Средним значением для числовых признаков.
    2. Удаляет признаки, у которых одно значение занимает 100% строк.
    """
    df_cleaned = df.copy()
    n_rows = len(df_cleaned)
    
    if n_rows == 0:
        return df_cleaned

    # Задаем список меток для пропущенных значений по умолчанию, если не передан свой
    if missing_values is None:
        missing_values = ['', ' ', 'N/A', 'NA', 'null', 'nan', 'NaN', 'None', 'none', '?', '-']

    # 1. Приведение пользовательских пропусков к np.nan и их импутация
    for col in df_cleaned.columns:
        # Заменяем все кастомные метки пропусков на np.nan
        df_cleaned[col] = df_cleaned[col].replace(missing_values, np.nan)

        if df_cleaned[col].isna().sum() > 0:
            if pd.api.types.is_numeric_dtype(df_cleaned[col]):
                # а) Числовые переменные -> средним значением
                mean_val = df_cleaned[col].mean()
                df_cleaned[col] = df_cleaned[col].fillna(mean_val)
            else:
                # б) Категориальные переменные -> модой
                mode_val = df_cleaned[col].mode()
                if not mode_val.empty:
                    df_cleaned[col] = df_cleaned[col].fillna(mode_val[0])

    # 2. Проверка условий для удаления колонок
    for col in df_cleaned.columns:
        series = df_cleaned[col]
        max_value_count = series.value_counts().iloc[0]
        max_value_ratio = max_value_count / n_rows

        # Если одинаковых значений = 100% -> к удалению
        if max_value_ratio == 1:
            cols_to_drop.append(col)

    # Удаляем выявленные колонки
    if cols_to_drop:
        df_cleaned = df_cleaned.drop(columns=cols_to_drop)
    return df_cleaned

df = clean_dataset(df, ["Unnamed: 0"])

In [23]:
df.isnull().sum()

acc_status                0
acc_duration              0
credit_history            0
purpose                   0
credit_amount             0
savings_acc               0
employment_time           0
installment_rate          0
gender                    0
other_debtors             0
residence_time            0
property                  0
age                       0
installment_plans         0
housing                   0
num_credits_at_bank       0
job                       0
num_people_maintenance    0
phone                     0
foreigner                 0
target                    0
dtype: int64

In [25]:
column_types = {
        "sufficient_resources_nominal_columns" : ['acc_status','credit_history','purpose','savings_acc',
                    'employment_time','gender','other_debtors', 'property',
                    'installment_plans','housing', 'foreigner'],
        "insufficient_resources_nominal_columns": None,
        "binary": ["phone"],
        "ordinal": None,
        "inclusion": None,
        "similarity":["job"],
    }
ANOC_threshold = 100

In [27]:
df['target'] = df['target'].map({1 : 1, 2 : 0 })
y = df["target"]
X = df.drop(["target"], axis=1)

In [33]:
import time
from sklearn.model_selection import StratifiedKFold, cross_validate, RepeatedStratifiedKFold
from sklearn.base import clone
from meta_encoder import MetaEncoder

# models
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

#from approach import encode_with_approach

model_types = {
    "LogReg": "ATI",
    "MLP": "ATI",
    "LinearSVM": "ATI",
    "CART": "tree",
    "RF": "tree",
    "CatBoost": "tree",
    "XGBoost": "tree",
    "LightGBM": "tree"
}
target_encoders = ['Leave-one-out', 'K-fold', 'CatBoost', 'CV-GLMM', 'WOE']

nominal_cols = column_types["sufficient_resources_nominal_columns"].copy()
if column_types["insufficient_resources_nominal_columns"]:
    nominal_cols.extend(column_types["insufficient_resources_nominal_columns"])

max_categories = df[nominal_cols].nunique().max()
ANOC = df.shape[0] / max_categories
# ANOC = 101

encode_with_target: bool = False

models = {
    "LogReg": LogisticRegression(max_iter=1000),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), activation="identity", max_iter=1000, random_state=42),
    "LinearSVM": LinearSVC(dual="auto", random_state=42),
    "CART": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(random_state=42, n_jobs=4),
    "XGBoost": XGBClassifier(eval_metric="logloss", n_jobs=4),
    "LightGBM": LGBMClassifier(verbose=-1, n_jobs=4),
    "CatBoost": CatBoostClassifier(verbose=0, thread_count=4)
}

records = []

# Инициализируем генератор разбиений
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

for model_name, model_template in models.items():

    # initializing guide parameters
    if model_types[model_name] == 'ATI':
        if ANOC > ANOC_threshold:
            encode_with_target = False
        else:
            encode_with_target = True
    else:  # 'tree-based'
        if ANOC > ANOC_threshold:
            encode_with_target = True
        else:
            encode_with_target = False

    target_encoders_to_apply = []

    if encode_with_target:
        target_encoders_to_apply = target_encoders
    else: # it just a mark that we use target-agnostic encoders
        target_encoders_to_apply = ["OneHot"]

    for nominal_encoder in target_encoders_to_apply:

        # creating parameters for the meta-encoder
        encode_columns_as = {}
        for type, columns in column_types.items():
            if type == "binary":
                encode_columns_as["Binary"] = columns
                continue
            if type == "similarity":
                encode_columns_as["Similarity"] = columns
                continue
            if type == "ordinal":
                encode_columns_as["Ordinal"] = columns
                continue
            if type == "inclusion":
                encode_columns_as["RankHot"] = columns
                continue

        encoder_name: str
        if not encode_with_target:
            if column_types["sufficient_resources_nominal_columns"] :
                encode_columns_as["OneHot"] = column_types["sufficient_resources_nominal_columns"]
                encoder_name = "OneHot"
            if column_types["insufficient_resources_nominal_columns"] :
                encode_columns_as["Hash"] = column_types["insufficient_resources_nominal_columns"]
                if encoder_name != '':
                    encoder_name = encoder_name + "_Hash"
        else:
            encode_columns_as[nominal_encoder] = nominal_cols
            encoder_name = nominal_encoder

        encoder = MetaEncoder(encode_columns_as=encode_columns_as)

        split_idx = 0
        # Проходим по всем фолдам вручную
        for train_idx, test_idx in cv.split(X, y):
            start_time = time.time()

            # 1. Разбиение выборки на train и test
            X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # 2. Инициализация и обработка кодировщиком
            X_train_encoded = encoder.fit_transform(X_train, y_train)
            X_test_encoded = encoder.transform(X_test)

            # 3. Масштабирование (если требуется для модели)
            if model_name in ["LogReg", "LinearSVM", "MLP"]:
                scaler = StandardScaler()
                X_train_processed = scaler.fit_transform(X_train_encoded)
                X_test_processed = scaler.transform(X_test_encoded)
            else:
                X_train_processed = X_train_encoded
                X_test_processed = X_test_encoded

            # 4. Обучение модели (клонируем или создаем заново, чтобы сбросить состояние)
            model = clone(model_template)
            model.fit(X_train_processed, y_train)

            # 5. Получение предсказаний
            y_pred = model.predict(X_test_processed)

            # Для ROC AUC и PR AUC нужны вероятности или decision_function
            if hasattr(model, "predict_proba"):
                y_proba = model.predict_proba(X_test_processed)[:, 1]
            elif hasattr(model, "decision_function"):
                y_proba = model.decision_function(X_test_processed)
            else:
                y_proba = y_pred

            # 6. Расчет метрик качество (scoring)
            auc_score = roc_auc_score(y_test, y_proba)
            pr_auc_score = average_precision_score(y_test, y_proba)
            acc_score = accuracy_score(y_test, y_pred)
            f1_val = f1_score(y_test, y_pred)
            prec_val = precision_score(y_test, y_pred)
            rec_val = recall_score(y_test, y_pred)

            elapsed_time = time.time() - start_time

            # 7. Вычисление номера повтора (repeat) и фолда (fold)
            repeat = split_idx // 5 + 1
            fold = split_idx % 5 + 1

            records.append(
                {
                    "approach": "Approach",
                    "dataset": "GermanCredit",
                    "model": model_name,
                    "encoder": encoder_name,
                    "repeat": repeat,
                    "fold": fold,
                    "auc": auc_score,
                    "pr_auc": pr_auc_score,
                    "accuracy": acc_score,
                    "f1": f1_val,
                    "precision": prec_val,
                    "recall": rec_val,
                    "total_time ": elapsed_time,
                }
            )

            print(
                f"Approach, GermanCredit, {model_name}, {encoder_name}, {repeat}, {fold}, "
                f"{auc_score}, {pr_auc_score}, {acc_score}, {f1_val}, "
                f"{prec_val}, {rec_val}, {elapsed_time}"
            )

            split_idx += 1
results = pd.DataFrame(records)
results.to_csv("results_german_credit_approach.csv", index=True)
print("Done!")

Approach, GermanCredit, LogReg, Leave-one-out, 1, 1, 0.7310714285714286, 0.8680198713607352, 0.7, 0.8013245033112583, 0.7469135802469136, 0.8642857142857143, 0.1269078254699707
Approach, GermanCredit, LogReg, Leave-one-out, 1, 2, 0.7851190476190476, 0.8813569473095824, 0.785, 0.8552188552188552, 0.8089171974522293, 0.9071428571428571, 0.09029626846313477
Approach, GermanCredit, LogReg, Leave-one-out, 1, 3, 0.7760714285714285, 0.8794349788236941, 0.745, 0.8235294117647058, 0.7986577181208053, 0.85, 0.08754324913024902
Approach, GermanCredit, LogReg, Leave-one-out, 1, 4, 0.7860714285714285, 0.8976976333120368, 0.73, 0.8187919463087249, 0.7721518987341772, 0.8714285714285714, 0.09369969367980957
Approach, GermanCredit, LogReg, Leave-one-out, 1, 5, 0.8094047619047617, 0.9006828199150209, 0.765, 0.8395904436860068, 0.803921568627451, 0.8785714285714286, 0.09382987022399902
Approach, GermanCredit, LogReg, Leave-one-out, 2, 1, 0.75, 0.8845186096520705, 0.72, 0.8120805369127517, 0.765822784810